# 1. Properties and traits

A **property** is a named axis of a compartmental model: disease state, age
band, vaccination status, strain. A **trait** is one value on that axis. Traits
within a property are mutually exclusive — a compartment has exactly one age
band, or none at all.

This chapter covers declaring properties, the trait objects they produce, and
the validation rules that catch typos before they reach an index array.

## Declaring a property

`Property(name, traits)` takes a name and a tuple of trait names. Declaration
order is meaningful: trait codes are `0 .. len(traits) - 1` in that order, and
that order determines compartment ordering later.

In [ ]:
from summer4 import Property

state = Property("state", ("S", "I", "R"))

assert state.name == "state"
assert state.traits == ("S", "I", "R")
state

Properties are frozen dataclasses compared **by value**, not identity. Two
properties with the same name and traits are equal, which makes them safe to
rebuild in a helper function rather than threading one object everywhere.

In [ ]:
assert Property("state", ("S", "I", "R")) == state
assert Property("state", ("S", "I")) != state

## Traits are selector leaves

`property.trait(name)` returns a `Trait`: the property name, the trait name and
its integer code. A `Trait` is not only a label — it is the smallest selector
you can write, and it composes with `&`, `|` and `~`.

In [ ]:
infectious = state.trait("I")

assert infectious.property == "state"
assert infectious.name == "I"
assert infectious.code == 1  # S=0, I=1, R=2

infectious

`state["I"]` is shorthand for `state.trait("I")`, and is how you will normally
write it.

In [ ]:
assert state["I"] == infectious

## Selecting several traits at once

Passing a sequence to `[]` — or calling `isin` explicitly — builds an `IsIn`
selector covering more than one trait. This is a single node, not a chain of
`|`, so it evaluates in one pass.

In [ ]:
from summer4 import IsIn

age = Property("age", ("0-4", "5-9", "10-19", "20+"))

children = age[("0-4", "5-9")]
assert isinstance(children, IsIn)
assert children == age.isin(["0-4", "5-9"])

children

## Presence and absence

Two more selectors come from the property itself. They matter once a property is
applied to only part of the compartment space — see {doc}`04-ragged-stratification`.

In [ ]:
severity = Property("severity", ("mild", "severe"))

print(severity.present())
print(severity.absent())

## Validation

Malformed properties raise immediately, at declaration, rather than producing a
confusing index error much later.

In [ ]:
for bad, reason in [
    (lambda: Property("", ("a",)), "empty property name"),
    (lambda: Property("p", ()), "no traits"),
    (lambda: Property("p", ("a", "")), "empty trait name"),
    (lambda: Property("p", ("a", "a")), "duplicate traits"),
]:
    try:
        bad()
    except ValueError as exc:
        print(f"{reason:<22} -> {exc}")
    else:
        raise AssertionError(f"expected {reason} to raise")

Unknown trait names raise `KeyError` and the message lists what *is* available,
which is the common case when a stratification is renamed.

In [ ]:
try:
    age["0-5"]
except KeyError as exc:
    print(exc)

## Selectors are not booleans

A selector describes a query; it has no truth value until it is resolved against
a compartment table. Using one in an `if` is almost always a mistake — chaining
`and`/`or` instead of `&`/`|` — so it raises.

In [ ]:
try:
    if state["I"]:
        pass
except TypeError as exc:
    print(exc)

---

Next: {doc}`02-building-a-property-map` turns these declarations into an actual
table of compartments.